In [ ]:
import os

# Task 1: Dataset directory and hyperparameters (must match training)
dataset_path = "./images_dataSAT"  # or os.path.join(data_dir, "images_dataSAT")

# Dataloader hyperparameters
img_w, img_h = 64, 64
batch_size = 128
num_classes = 2
agri_class_labels = ["non-agri", "agri"]

# PyTorch CNN-ViT Hybrid model hyperparameters
depth = 3
attn_heads = 6
embed_dim = 768

print("Dataset path:", dataset_path)
print("Dataloader hyperparameters:")
print(f"  Image size: {img_w}x{img_h}")
print(f"  Batch size: {batch_size}")
print(f"  Num classes: {num_classes}")
print("\nHybrid CNN-ViT Model Hyperparameters:")
print(f"  Depth: {depth}")
print(f"  Attention heads: {attn_heads}")
print(f"  Embedding dim: {embed_dim}")

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# Define the model architecture (CNN + ViT Hybrid)
class PatchEmbed(nn.Module):
    def __init__(self, in_chans=128, embed_dim=768, patch_size=16):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=768, num_heads=8, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class CNN_ViT_Hybrid(nn.Module):
    def __init__(self, num_classes=2, heads=6, depth=3, embed_dim=768, patch_size=16):
        super().__init__()
        self.cnn = nn.Sequential(  # Simple CNN feature extractor
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.patch_embed = PatchEmbed(in_chans=128, embed_dim=embed_dim, patch_size=patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, 1000, embed_dim))
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, heads) for _ in range(depth)])
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.patch_embed(x)
        x = x + self.pos_embed[:, :x.size(1), :]
        for blk in self.blocks:
            x = blk(x)
        x = x.mean(dim=1)  # or x[:, 0] for CLS
        x = self.head(x)
        return x

# Instantiate
device = "cuda" if torch.cuda.is_available() else "cpu"
pytorch_model = CNN_ViT_Hybrid(
    num_classes=num_classes,
    heads=attn_heads,
    depth=depth,
    embed_dim=embed_dim
).to(device)

print(f"PyTorch model instantiated on {device}")

In [ ]:
pytorch_state_dict_path = "./pytorch_cnn_vit_ai_capstone_model_state_dict.pth"
pytorch_model.load_state_dict(torch.load(pytorch_state_dict_path, map_location=device), strict=False)
pytorch_model.eval()

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, log_loss, 
                             classification_report, confusion_matrix, 
                             ConfusionMatrixDisplay)
import numpy as np
import matplotlib.pyplot as plt

def print_metrics(y_true, y_pred, y_prob, class_labels, model_name):
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob[:,1] if y_prob.ndim > 1 else y_prob),
        'Loss': log_loss(y_true, y_prob),
    }
    print(f"Evaluation metrics for the **{model_name}**")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
    print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=class_labels))
    # Confusion Matrix
    disp = ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=class_labels)
    disp.plot()
    plt.show()

In [ ]:
# Assuming you already ran inference and have:
# all_labels_keras, all_preds_keras, all_probs_keras

print_metrics(
    y_true=all_labels_keras,
    y_pred=all_preds_keras,
    y_prob=all_probs_keras,
    class_labels=agri_class_labels,
    model_name="Keras CNN-ViT Hybrid Model"
)

In [ ]:
# Run inference if not done
all_labels_pytorch = []
all_preds_pytorch = []
all_probs_pytorch = []

with torch.no_grad():
    for images, labels in tqdm(test_loader):  # your DataLoader
        images = images.to(device)
        outputs = pytorch_model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)
        
        all_labels_pytorch.extend(labels.cpu().numpy())
        all_preds_pytorch.extend(preds.cpu().numpy())
        all_probs_pytorch.extend(probs.cpu().numpy())

print_metrics(
    y_true=np.array(all_labels_pytorch),
    y_pred=np.array(all_preds_pytorch),
    y_prob=np.array(all_probs_pytorch),
    class_labels=agri_class_labels,
    model_name="PyTorch CNN-ViT Hybrid Model"
)